In [ ]:
from pathlib import Path
import hashlib,json,os,signal,subprocess,sys,tarfile,time,traceback,urllib.request,zipfile
import psutil
RUN_ID='__RUN_ID__'
WORK=Path('/kaggle/working')/'runs'/'b319'/RUN_ID;WORK.mkdir(parents=True,exist_ok=False)
TEMP=Path('/kaggle/tmp')/RUN_ID;TEMP.mkdir(parents=True,exist_ok=False)
RECEIPTS=WORK/'receipts';RECEIPTS.mkdir()
SETUP=WORK/'logs'/'setup';SETUP.mkdir(parents=True,exist_ok=False)
START=time.monotonic();DEADLINE=START+30600
status={'status':'SETUP','source_sha256':'47d41588c488dd9668c5ac5747f8ffbddd80df5f7d9b0a016ce709062f71e4b1','arms':['er_uniform','a0_tail','a0_tail_ctx'],'scientific_acceptance':False}
def save():
    (RECEIPTS/'RUN_STATUS.json').write_text(json.dumps(status,indent=2)+'\n')
def command(args,name,seconds,ram=False):
    limit=min(DEADLINE-120,time.monotonic()+seconds)
    print('COMMAND_START',name,flush=True)
    env=os.environ.copy();env.update(PYTHONUNBUFFERED='1',PYTHONDONTWRITEBYTECODE='1',PYTHONPATH=str(ROOT/'src'),CUBLAS_WORKSPACE_CONFIG=':4096:8',XFORMERS_DISABLED='1',OMP_NUM_THREADS='4',MKL_NUM_THREADS='4',NRCAD_LOCAL_ROOT=str(TEMP/'local'))
    log=SETUP/(name+'.log')
    with log.open('w') as output,log.open() as reader:
        proc=subprocess.Popen([str(x) for x in args],cwd=ROOT,env=env,stdout=output,stderr=subprocess.STDOUT,start_new_session=True)
        try:
            while proc.poll() is None:
                text=reader.read()
                if text:print(text,end='',flush=True)
                if time.monotonic()>limit:raise TimeoutError(name+' cap exceeded')
                if ram:
                    try:
                        parent=psutil.Process(proc.pid);rss=sum(p.memory_info().rss for p in [parent,*parent.children(recursive=True)] if p.is_running())
                        if rss>8*1024**3:raise MemoryError(name+' process-group RSS cap8GiB exceeded')
                    except psutil.NoSuchProcess:pass
                time.sleep(1)
            text=reader.read()
            if text:print(text,end='',flush=True)
            if proc.returncode:raise RuntimeError(name+' exit='+str(proc.returncode))
        finally:
            if proc.poll() is None:
                os.killpg(proc.pid,signal.SIGTERM)
                try:proc.wait(timeout=10)
                except subprocess.TimeoutExpired:os.killpg(proc.pid,signal.SIGKILL);proc.wait()
    print('COMMAND_DONE',name,flush=True)
def load_source(input_root,destination,zip_sha,manifest_sha):
    import shutil
    candidates=sorted(input_root.rglob('source.zip'))
    valid=[p for p in candidates if hashlib.sha256(p.read_bytes()).hexdigest()==zip_sha]
    if destination.exists():raise FileExistsError('Source destination already exists')
    destination.mkdir(parents=True)
    if valid:
        with zipfile.ZipFile(valid[0]) as archive:
            for info in archive.infolist():
                if not (destination/info.filename).resolve().is_relative_to(destination.resolve()):
                    raise ValueError('Unsafe source path')
            archive.extractall(destination)
        layout='zip'
    else:
        bundles=sorted(input_root.rglob('BUNDLE.json'))
        valid_bundles=[p for p in bundles if hashlib.sha256(p.read_bytes()).hexdigest()==manifest_sha]
        if not valid_bundles:
            raise ValueError('No pinned source found: ZIP candidates='+str(len(candidates))+'; expanded manifests='+str(len(bundles)))
        bundle=valid_bundles[0];manifest=json.loads(bundle.read_bytes())
        for name,expected in manifest.items():
            target=destination/name
            if not target.resolve().is_relative_to(destination.resolve()):raise ValueError('Unsafe manifest path')
            source=bundle.parent/name
            if hashlib.sha256(source.read_bytes()).hexdigest()!=expected:raise ValueError('Mounted source mismatch '+name)
            target.parent.mkdir(parents=True,exist_ok=True);shutil.copyfile(source,target)
        shutil.copyfile(bundle,destination/'BUNDLE.json');layout='expanded'
    raw=(destination/'BUNDLE.json').read_bytes()
    if hashlib.sha256(raw).hexdigest()!=manifest_sha:raise ValueError('Pinned source manifest mismatch')
    manifest=json.loads(raw)
    for name,expected in manifest.items():
        if not (destination/name).resolve().is_relative_to(destination.resolve()):raise ValueError('Unsafe manifest path')
        if hashlib.sha256((destination/name).read_bytes()).hexdigest()!=expected:raise ValueError('Source member mismatch '+name)
    return manifest,layout

try:
    save();print('B319_STANDALONE_BOOTSTRAP',json.dumps(status),flush=True)
    ROOT=TEMP/'source'
    bundle,layout=load_source(Path('/kaggle/input'),ROOT,'47d41588c488dd9668c5ac5747f8ffbddd80df5f7d9b0a016ce709062f71e4b1','003b91932dbd6713782f0fd90f64dbdedcea12ab53923fe6a31e2a8dcc06f2f5')
    status['source_layout']=layout
    print('SOURCE_LAYOUT',layout,flush=True)
    print('VERIFIED_SOURCE',len(bundle),'files',flush=True)
    (RECEIPTS/'source_manifest.json').write_text(json.dumps(bundle,indent=2)+'\n')
    pixi_tar=TEMP/'pixi.tar.gz'
    urllib.request.urlretrieve('https://github.com/prefix-dev/pixi/releases/download/v0.69.0/pixi-x86_64-unknown-linux-musl.tar.gz',pixi_tar)
    bin_dir=TEMP/'bin';bin_dir.mkdir()
    with tarfile.open(pixi_tar) as archive:archive.extractall(bin_dir,filter='data')
    PIXI=bin_dir/'pixi';PIXI.chmod(0o755)
    command([PIXI,'--version'],'pixi-version',60)
    command([PIXI,'install','--locked','-e','cad'],'install',2400)
    python=ROOT/'.pixi/envs/cad/bin/python'
    assets=TEMP/'local'
    command([python,'-B','scripts/prepare_assets.py','--root',assets],'assets',1800,True)
    common=['--data',assets/'inputs/mvtec','--weights',assets/'inputs/weights/dinov2_vitb14_reg4_pretrain.pth']
    command([python,'-B','scripts/train.py','--check',*common],'inputs',600,True)
    status['status']='GPU_PREFLIGHT';save()
    command([python,'-B','scripts/preflight.py',*common,'--output',RECEIPTS/'gpu_preflight.json'],'gpu-preflight',600,True)
    status['status']='TRAINING';save()
    command([python,'-B','scripts/train.py',*common,'--output',WORK/'outputs','--arm','all'],'b319-three-arms',int(DEADLINE-time.monotonic()-120),True)
    status['status']='COMPLETE_REVIEW_PENDING'
except BaseException as error:
    status.update(status='FAILED',error=repr(error),traceback=traceback.format_exc());raise
finally:
    status['seconds']=time.monotonic()-START;save();print('B319_TERMINAL',json.dumps(status),flush=True)
